# Training

In [43]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [45]:
import time

from pathlib import Path
from datasets import load_from_disk
from tqdm.notebook import tqdm

import sagemaker

from config.settings import AWSSettings, DatasetSettings
from sagemaker.huggingface import HuggingFace
from sagemaker.debugger import TensorBoardOutputConfig
from sagemaker.huggingface.model import HuggingFaceModel

# SageMaker INIT

In [4]:
aws_settings = AWSSettings()
dataset_settings = DatasetSettings()

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Get the AWS region
region = sagemaker_session.boto_region_name
print(f"SageMaker running in region: {region}")
print(f"SageMaker role ARN: {aws_settings.EXECUTION_ROLE}")
print(f"SageMaker bucket: {aws_settings.BUCKET}")

[03/24/25 18:42:39] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=220265;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=243418;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

SageMaker running in region: eu-central-1
SageMaker role ARN: arn:aws:iam::421646001410:role/service-role/AmazonSageMaker-ExecutionRole-20210811T103532
SageMaker bucket: pivanov-tac-bucket


## Fine-tune RoBERTa

In [38]:
model_name = "roberta-base"

hyperparameters = {
    "epochs": 15,
    "train_batch_size": 8,
    "eval_batch_size": 8,
    "warmup_steps": 500,
    "learning_rate": 5e-5,
    }

tensorboard_output_config = TensorBoardOutputConfig(
    s3_output_path=f"s3://{aws_settings.BUCKET}/tensorboard/",
    container_local_output_path="/opt/ml/output/tensorboard"
)

In [42]:
metric_definitions = [
    {"Name": "eval_loss", "Regex": r"'eval_loss': ([0-9\\.]+)"},
    {"Name": "eval_accuracy", "Regex": r"'eval_accuracy': ([0-9\\.]+)"},
    {"Name": "eval_f1", "Regex": r"'eval_f1': ([0-9\\.]+)"},
    {"Name": "eval_precision", "Regex": r"'eval_precision': ([0-9\\.]+)"},
    {"Name": "eval_recall", "Regex": r"'eval_recall': ([0-9\\.]+)"},
]

huggingface_estimator = HuggingFace(
        entry_point="train-roberta.py",
        instance_type="ml.g4dn.2xlarge",
        instance_count=1,
        role=aws_settings.EXECUTION_ROLE,
        output_path=f"s3://{aws_settings.BUCKET}/output/",
        metric_definitions=metric_definitions,
        transformers_version="4.6.1",
        pytorch_version="1.7.1",
        py_version="py36",
        hyperparameters = hyperparameters
)

In [43]:
training_parameters = {
    "train": f"s3://{aws_settings.BUCKET}/datasets/train/",
    "test": f"s3://{aws_settings.BUCKET}/datasets/test/",
}

In [45]:
# Run
huggingface_estimator.fit(
    inputs=training_parameters,
    job_name=f"tac-sentiment-analysis-{model_name}-{time.strftime('%Y-%m-%d-%H-%M', time.gmtime())}",
    wait=False
)

[03/24/25 15:38:20] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=593858;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=39950;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=865968;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=409533;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/image_uris.py#681\681]8;;\
                             instance_type, framework etc.                                                         

                    INFO     Creating training-job with name:                                       ]8;id=833905;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=504037;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             tac-sentiment-analysis-roberta-base-2025-03-24-14-38                                  

## Fine-tune TinyBERT

In [28]:
model_name = "tinybert"

hyperparameters = {
    "epochs": 15,
    "train_batch_size": 8,
    "eval_batch_size": 8,
    "warmup_steps": 500,
    "learning_rate": 5e-5,
    }

tensorboard_output_config = TensorBoardOutputConfig(
    s3_output_path=f"s3://{aws_settings.BUCKET}/tensorboard/",
    container_local_output_path="/opt/ml/output/tensorboard"
)

In [29]:
metric_definitions = [
    {"Name": "eval_loss", "Regex": r"'eval_loss': ([0-9\\.]+)"},
    {"Name": "eval_accuracy", "Regex": r"'eval_accuracy': ([0-9\\.]+)"},
    {"Name": "eval_f1", "Regex": r"'eval_f1': ([0-9\\.]+)"},
    {"Name": "eval_precision", "Regex": r"'eval_precision': ([0-9\\.]+)"},
    {"Name": "eval_recall", "Regex": r"'eval_recall': ([0-9\\.]+)"},
]

huggingface_estimator = HuggingFace(
        entry_point="train-tinybert.py",
        instance_type="ml.g4dn.2xlarge",
        tensorboard_output_config=tensorboard_output_config,
        instance_count=1,
        role=aws_settings.EXECUTION_ROLE,
        output_path=f"s3://{aws_settings.BUCKET}/output/",
        metric_definitions=metric_definitions,
        transformers_version="4.6.1",
        pytorch_version="1.7.1",
        py_version="py36",
        hyperparameters = hyperparameters
)

In [30]:
training_parameters = {
    "train": f"s3://{aws_settings.BUCKET}/datasets/train/",
    "test": f"s3://{aws_settings.BUCKET}/datasets/test/",
}

In [33]:
# Run
huggingface_estimator.fit(
    inputs=training_parameters,
    job_name=f"tac-sentiment-analysis-{model_name}-{time.strftime('%Y-%m-%d-%H-%M', time.gmtime())}",
    wait=False
)

[03/24/25 11:34:16] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=245793;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10185;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     image_uri is not presented, retrieving image_uri based on            ]8;id=234103;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/image_uris.py\image_uris.py]8;;\:]8;id=699139;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/image_uris.py#681\681]8;;\
                             instance_type, framework etc.                                                         

                    INFO     Creating training-job with name:                                       ]8;id=364380;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=937088;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             tac-sentiment-analysis-tinybert-2025-03-24-10-34                                      

In [61]:
def relabel(func, mapping: dict):
    """
    Relabel the output of a function using a mapping."
    """
    def wrapper(data):
        result = func(data)
        result["label"] = mapping[result["label"]]
        return result
    return wrapper

In [ ]:
@relabel(predictor.predict, {0: "neg", 1: "pos"})
def predict_sentiment(data, predictor):
    """
    Predict the sentiment of a given text.
    """
    result = predictor.predict(data)
    return result

In [ ]:
# example request: you always need to define "inputs"
data = {
   "inputs": "Camera - You are awarded a SiPix Digital Camera! call 09061221066 fromm landline. Delivery within 28 days."
}

# request
predictor.predict(
    data,
    initial_args={"ContentType": "application/json"},
)



[{'label': 'LABEL_1', 'score': 0.9111132025718689}]

In [8]:
from sagemaker.huggingface.model import HuggingFacePredictor

# Load the predictor
predictor = HuggingFacePredictor(
    endpoint_name="tac-sentiment-analysis-tinybert-2025-03-24-16-05"
)

# Example evaluation
data = {
   "inputs": "Camera - You are awarded a SiPix Digital Camera! call 09061221066 fromm landline. Delivery within 28 days."
}

# Request
response = predictor.predict(
    data,
    initial_args={"ContentType": "application/json"},
)

print(response)

[03/24/25 21:24:13] INFO     Found credentials in shared credentials file: ~/.aws/credentials   ]8;id=923036;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=570957;file:///home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/credentials.py#1352\1352]8;;\

[{'label': 'LABEL_1', 'score': 0.9111132025718689}]


In [9]:
test_dataset = load_from_disk("test_dataset")
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})


In [30]:
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
from transformers import RobertaTokenizer

# Extract texts and labels from the test dataset
texts = test_dataset['text']
true_labels = test_dataset['label']

# Get predictions from the predictor
# Initialize the tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Truncate texts to 512 tokens
texts = [tokenizer.decode(tokenizer.encode(text, max_length=512, truncation=True), skip_special_tokens=True) for text in test_dataset['text']]

In [36]:
len(tokenizer.encode(texts[9], max_length=512, truncation=True))

247

In [37]:
predicted_labels = []
for text in tqdm(texts, desc="Predicting"):
    response = predictor.predict({"inputs": text}, initial_args={"ContentType": "application/json"})
    predicted_labels.append(response[0]['label'])
    # Map responses from predictor
    try:
        response = predictor.predict({"inputs": text}, initial_args={"ContentType": "application/json"})
        predicted_labels.append(response[0]['label'])
    except Model as e:
        print(f"Error predicting for text: {text}")
        print(f"Exception: {e}")

label_mapping = {"LABEL_1": 1, "LABEL_0": 0}
predicted_labels = [label_mapping[label] for label in predicted_labels]
# Calculate metrics
accuracy = accuracy_score(true_labels, predicted_labels)
recall = recall_score(true_labels, predicted_labels, pos_label=1)
f1 = f1_score(true_labels, predicted_labels, pos_label=1)
auc = roc_auc_score(true_labels, predicted_labels)

# Print metrics
print(f"Accuracy: {accuracy}")
print(f"Recall: {recall}")
print(f"F1 Score: {f1}")
print(f"AUC: {auc}")

Predicting:   0%|          | 0/25000 [00:00<?, ?it/s]

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│    1 predicted_labels = []                                                                       │
│    2 for text in tqdm(texts, desc="Predicting"):                                                 │
│ ❱  3 │   response = predictor.predict({"inputs": text}, initial_args={"ContentType": "applica    │
│    4 │   predicted_labels.append(response[0]['label'])                                           │
│    5 │   # Map responses from predictor                                                          │
│    6 │   try:                                                                                    │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/base_predictor.py:212 in    │
│ predict                                                                                          │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/client.py:570 in _api_call   │
│                                                                                                  │
│    567 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    568 │   │   │   │   )                                                                         │
│    569 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  570 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    571 │   │                                                                                     │
│    572 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    573                                                                                           │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/context.py:124 in wrapper    │
│                                                                                                  │
│   121 │   │   │   with start_as_current_context():                                               │
│   122 │   │   │   │   if hook:                                                                   │
│   123 │   │   │   │   │   hook()                                                                 │
│ ❱ 124 │   │   │   │   return func(*args, **kwargs)                                               │
│   125 │   │                                                                                      │
│   126 │   │   return wrapper                                                                     │
│   127                                                      

In [28]:
tokenizer.decode(tokenizer.encode("Oh dude"), skip_special_tokens=True)

'Oh dude'

In [40]:
predictor.predict({"inputs": test_dataset[8]["text"]}, initial_args={"ContentType": "application/json"})

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 predictor.predict({"inputs": test_dataset[8]["text"]}, initial_args={"ContentType": "app     │
│   2                                                                                              │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/sagemaker/base_predictor.py:212 in    │
│ predict                                                                                          │
│                                                                                                  │
│   209 │   │   if inference_component_name:                                                       │
│   210 │   │   │   request_args["InferenceComponentName"] = inference_component_name              │
│   211 │   │                                                                                      │
│ ❱ 212 │   │   response = self.sagemaker_session.sagemaker_runtime_client.invoke_endpoint(**req   │
│   213 │   │   return self._handle_response(response)                                             │
│   214 │                                                                                          │
│   215 │   def _handle_response(self, response):                                                  │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/client.py:570 in _api_call   │
│                                                                                                  │
│    567 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    568 │   │   │   │   )                                                                         │
│    569 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  570 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    571 │   │                                                                                     │
│    572 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    573                                                                                           │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/context.py:124 in wrapper    │
│                                                                                                  │
│   121 │   │   │   with start_as_current_context():                                               │
│   122 │   │   │   │   if hook:                                                                   │
│   123 │   │   │   │   │   hook()                                                                 │
│ ❱ 124 │   │   │   │   return func(*args, **kwargs)                                               │
│   125 │   │                                                                                      │
│   126 │   │   return wrapper                                                                     │
│   127                                                                                            │
│                                                                                                  │
│ /home/ssm-user/code/gda/.venv/lib/python3.10/site-packages/botocore/client.py:1031 in            │
│ _make_api_call                                                                                   │
│                                                            

In [14]:
true_labels[:5]

[1, 1, 1, 1, 1]

In [17]:
data = {
   "inputs": "Didn't like it"
}

In [18]:
predictor.predict(
    data,
    initial_args={"ContentType": "application/json"},
)

[{'label': 'LABEL_0', 'score': 0.9714904427528381}]